In [2]:
import numpy as np
import pandas as pd

In [3]:
df=pd.read_csv('train.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [6]:
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
225,0,3,male,22.0,0,0,9.3500,S
302,0,3,male,19.0,0,0,0.0000,S
343,0,2,male,25.0,0,0,13.0000,S
197,0,3,male,42.0,0,1,8.4042,S
336,0,1,male,29.0,1,0,66.6000,S


In [7]:
from sklearn.model_selection import train_test_split

x=df.drop(columns=['Survived'])
y=df['Survived']

x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=0
)

In [8]:
x_train.shape,x_test.shape

((712, 7), (179, 7))

In [9]:
y_train.sample(5)

537    1
430    1
598    0
337    1
436    0
Name: Survived, dtype: int64

In [11]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,3,female,NaN,0,2,15.2458,C
439,2,male,31.0,0,0,10.5000,S
817,2,male,31.0,1,1,37.0042,C
378,3,male,20.0,0,0,4.0125,C
491,3,male,21.0,0,0,7.2500,S


In [12]:
x_train.isnull().sum()

Pclass        0
Sex           0
Age         141
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

## Handling NULL values

In [14]:
## imputation Transformer / NULL Value Handling
trf1=ColumnTransformer(
    [
        ('impute_age',SimpleImputer(),[2]),
        ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])         ### 2 and 6 is index of age and embarked 
    ],
    remainder='passthrough'          
)

## Handling Categorical data

In [45]:
## onehotEncoding on sex and embarked
trf2=ColumnTransformer(
    [
        ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])  ## 1 and 6 is index of sex and embarked
    ],
    remainder='passthrough'
    )
                                           

In [46]:
from sklearn.preprocessing import MinMaxScaler

In [47]:
## Scaling 
trf3=ColumnTransformer(
    [('scaling',MinMaxScaler(),slice(0,10))]
)

In [48]:
from sklearn.feature_selection import SelectKBest,chi2

In [49]:
# Feature selection
trf4 = SelectKBest(score_func=chi2,k=8)

In [50]:
from sklearn.tree import DecisionTreeClassifier

In [51]:
## train the model
trf5=DecisionTreeClassifier()

## Create Pipeline

In [52]:
from sklearn.pipeline import Pipeline

In [53]:
pipe=Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5),
])

# Pipeline VS make_pipeline

In [54]:
## Pipeline requires naming of steps, make_pipeline does not.

## (Same applies to ColumnTransformer vs make_column_transformer

In [56]:
from sklearn.pipeline import make_pipeline

In [57]:
## Alternate Syntex
pipe=make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [59]:
## Displaying pipeline
from sklearn import set_config
set_config(display='diagram')

In [61]:
## Checking score
pipe.fit(x_train,y_train).score(x_test,y_test)

0.6759776536312849

# Cross Validation using Pipeline

In [63]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, x_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.6152171771890081)

# Exporting the Pipeline

In [64]:
# export 
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))  ## pipe.pkl is file name nad wb is write in binary mode